In [88]:
# Juptyer Lab에서 Django shell을 실행하기 위한 설정

import os
import django

#환경 변수로 config/settings.py의 위치를 설정
os.environ["DJANGO_SETTINGS_MODULE"] = "config.settings"
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

django.setup()

In [89]:
# 조회 테스트
from polls.models import Question, Choice

q = Question.objects.all()
q

<QuerySet [<Question: 1. 가장 좋아하는 음식은 ?>, <Question: 2. 가장 좋아하는 날씨는 ?>]>

# 조회
- ModelClass.objects -> Model Manager를 반환.
- ModelManager: SQL작업을 할 수있는 메소드들을 제공하는 객체

## 조회메소드
- `all()`: 전체 조회
- `filter()`, `exclude()`: 조건으로 조회(where절)
- `get()`: 조회결과가 하나인 조건으로 조회(PK로 조회)

## 조회결과
- `QuerySet` 객체: 조회결과가 여러개일때 QuerySet에 모아서 반환.
  - 조회결과를 바탕으로 추가 DB 작업을 진행할 수 있는 메소드들을 제공.
  - 개별 데이터는 Model 객체에 담아서 반환.
- Model 객체: 조회결과가 하나(`get()`) 일 때 

In [6]:
from polls.models import Question, Choice

In [8]:
model_manager = Question.objects
type(model_manager)

django.db.models.manager.Manager

In [11]:
result = model_manager.all()
print("조회한 data 개수 :", len(result))
print("all()로 실행된 SQL문을 조회")
print(result.query)

조회한 data 개수 : 2
all()로 실행된 SQL문을 조회
SELECT "polls_question"."id", "polls_question"."question_text", "polls_question"."pub_date" FROM "polls_question"


In [12]:
print(type(result))

<class 'django.db.models.query.QuerySet'>


In [17]:
# QuerySet -> Iterable
for q in result:
    print(q, q.pub_date)

1. 가장 좋아하는 음식은 ? 2025-07-07 06:31:23.893928+00:00
2. 가장 좋아하는 날씨는? 2025-07-07 06:31:29.254495+00:00


In [21]:
# QuerySet -> Subscriptable
q = result[1]
q.id, q.pk, q.question_text, q.pub_date

(2,
 2,
 '가장 좋아하는 날씨는?',
 datetime.datetime(2025, 7, 7, 6, 31, 29, 254495, tzinfo=datetime.timezone.utc))

In [22]:
# 첫 번째, 마지막 idx값 조회
q_s = result.first()
q_e = result.last()

q_s.pk, q_e.pk

(1, 2)

In [ ]:
# slicing 결과는 list, 음수 indexing은 지원 X
s_result = result[:2]
s_result

[<Question: 1. 가장 좋아하는 음식은 ?>, <Question: 2. 가장 좋아하는 날씨는?>]

In [26]:
# QuerySet을 이용해서 정렬 (order by)
## QS.orderby("기준Field명") : ASC, QS.orderby("-기준Field명") : DESC,
print(result.order_by("question_text"))
print(result.order_by("-question_text"))

<QuerySet [<Question: 2. 가장 좋아하는 날씨는?>, <Question: 1. 가장 좋아하는 음식은 ?>]>
<QuerySet [<Question: 1. 가장 좋아하는 음식은 ?>, <Question: 2. 가장 좋아하는 날씨는?>]>


In [90]:
# Choice에 모든 data를 조회 -> question column 기준으로 정렬
results = Choice.objects.all()
for c in results:
    print(c.pk, c.choice_text, c.votes)

1 짜장미엔 3
2 탕수육 10
3 탕수육 2
4 짬뽕 13
5 고추잡채 2
6 화창~ 0
7 비 4


In [33]:
# order by choice_tet, votes desc
results = Choice.objects.all().order_by("choice_text", "-votes")
for c in results:
    print(c.pk, c.choice_text, c.votes)

5 고추잡채 1
1 짜장미엔 0
4 짬뽕 2
2 탕수육 5
3 탕수육 2


### Where 절
- **filter()**
	- 조회조건이 True인 행들을 조회 -> QuerySet 반환
- **exclude()**
	- 조회조건이 False인 행들을 조회 -> QuerySet 반환
- **get()**
	- 조회조건이 True인 행이 1개일 때 조회 -> Model에 결과를 담아서 반환.
	- 조회조건이 2행 이상이거나 없을 경우 Exception 발생
- **조회조건**
	- `Fielod이름__비교연산자 = 비교할 값`

In [40]:
# pk 조회 -> 결과 1행 | 0행
result = Question.objects.get(pk=1) 		# where id = 1, 동등 비교 : field명 = 값
result = Question.objects.filter(pk = 1)	# 조회 결과가 0개 이상
result = Question.objects.exclude(pk = 1)	# where not id = 1

print(type(result))
print(result)

<class 'django.db.models.query.QuerySet'>
<QuerySet [<Question: 2. 가장 좋아하는 날씨는?>]>


In [56]:
# 비교 엿나
result1 = Choice.objects.filter(pk__lt = 5)		# less then
result2 = Choice.objects.filter(pk__lte = 5)	# less then equal
result3 = Choice.objects.filter(pk__gt = 3)		# greater then
result4 = Choice.objects.filter(pk__gte = 3)	# greater then equal

print(result1, result4)


<QuerySet [<Choice: 1. 짜장미엔>, <Choice: 2. 탕수육>, <Choice: 3. 탕수육>, <Choice: 4. 짬뽕>]> <QuerySet [<Choice: 3. 탕수육>, <Choice: 4. 짬뽕>, <Choice: 5. 고추잡채>]>


In [58]:
# 문자열 부분 일치 - like (포함, 시작, 끝)
result = Question.objects.filter(question_text__contains="음식")

result = Question.objects.filter(question_text__startswith="가장")

result = Question.objects.filter(question_text__endswith="?")

for r in result:
    print(r)

1. 가장 좋아하는 음식은 ?
2. 가장 좋아하는 날씨는?


In [61]:
# in 연산
result = Choice.objects.filter(pk__in=[1, 3, 5])	# pk in (1, 3, 5)

result = Choice.objects.exclude(pk__in=[1, 3, 5])	# pk not in (1, 3, 5)

print(result.query)
for r in result:
    print(r)

SELECT "polls_choice"."id", "polls_choice"."choice_text", "polls_choice"."votes", "polls_choice"."question_id" FROM "polls_choice" WHERE NOT ("polls_choice"."id" IN (1, 3, 5))
2. 탕수육
4. 짬뽕


In [63]:
# between
# pk between 2 and 5
result = Choice.objects.filter(pk__range=[2, 5])
# pk not between 2 and 5
result = Choice.objects.exclude(pk__range=[2, 5])

print(result.query)
for r in result:
    print(r)

SELECT "polls_choice"."id", "polls_choice"."choice_text", "polls_choice"."votes", "polls_choice"."question_id" FROM "polls_choice" WHERE NOT ("polls_choice"."id" BETWEEN 2 AND 5)
1. 짜장미엔


### where 절의 and, or
- `and` : 조건들을 나열
	
- `or` : 각 조건을 `Q()` func에 넣고 `|`로 연결


In [65]:
result = Question.objects.filter(
    question_text__endswith = "은 ?",
    pk__lte=2
)



print(result.query)
for r in result:
    print(r)

SELECT "polls_question"."id", "polls_question"."question_text", "polls_question"."pub_date" FROM "polls_question" WHERE ("polls_question"."id" <= 2 AND "polls_question"."question_text" LIKE %은 ? ESCAPE '\')
1. 가장 좋아하는 음식은 ?


In [72]:
from django.db.models import Q
result = Question.objects.filter(
    Q(question_text__endswith = "은 ?") |
    Q(pk__lte = 2)
)

print(result.query)
for r in result:
    print(r)

SELECT "polls_question"."id", "polls_question"."question_text", "polls_question"."pub_date" FROM "polls_question" WHERE ("polls_question"."question_text" LIKE %은 ? ESCAPE '\' OR "polls_question"."id" <= 2)
1. 가장 좋아하는 음식은 ?
2. 가장 좋아하는 날씨는 ?


In [74]:
result = Question.objects.filter(
    ~Q(question_text__endswith = "은 ?") |
    ~Q(pk__lte = 2)
)

print(result.query)
for r in result:
    print(r)

SELECT "polls_question"."id", "polls_question"."question_text", "polls_question"."pub_date" FROM "polls_question" WHERE (NOT ("polls_question"."question_text" LIKE %은 ? ESCAPE '\') OR NOT ("polls_question"."id" <= 2))
2. 가장 좋아하는 날씨는 ?


### 조회할 column 선택
- `values(Field명, ...)`
- 개별 조회 결과를 dict로 반환

In [78]:
result = Question.objects.all().values("pk", "question_text")

for r in result:
    print(type(r), r["pk"], r["question_text"])

<class 'dict'> 1 가장 좋아하는 음식은 ?
<class 'dict'> 2 가장 좋아하는 날씨는 ?


In [79]:
result = Choice.objects.filter(pk__lt = 3).values("pk", "votes")

print(result.query)
result

SELECT "polls_choice"."id" AS "pk", "polls_choice"."votes" AS "votes" FROM "polls_choice" WHERE "polls_choice"."id" < 3


<QuerySet [{'pk': 1, 'votes': 0}, {'pk': 2, 'votes': 5}]>

### 집계 함수
- `aggregate(집계함수(집계기준field명), 집계함수(field명), ...)`
	- ex) `select avg(salary), count(comm_pct), max(salary) from emp`
- **groupby**
	- values("groupby 기준 컬럼").annotate(집계함수)

In [80]:
from django.db.models import (
    Count,		# 값의 개수 (NULL 제외)
    Sum,		# 합계
    Avg,
    Min, Max,
    StdDev,		# 표준 편차
    Variance	# 분산
)

In [83]:
result = Choice.objects.aggregate(
    Count("votes"),
    Sum("votes"),
    Avg("votes"),
    Min("votes"),Max("votes"),
    StdDev("votes"),
    Variance("votes")
)

print(result)
# 반환 : dict
# default key : field명__집계명

{'votes__count': 5, 'votes__sum': 10, 'votes__avg': 2.0, 'votes__min': 0, 'votes__max': 5, 'votes__stddev': 1.6733200530681511, 'votes__variance': 2.8}


In [85]:
result = Choice.objects.aggregate(
    cnt = Count("votes"),
    min = Min("votes"),
    max = Max("votes")
)

result

{'cnt': 5, 'min': 0, 'max': 5}

In [86]:
# 집계 결과를 연산
# 변수명 = (집계함수 - 집계함수)
Choice.objects.aggregate(min_max_diff = (Max("votes") - Min("votes")))

{'min_max_diff': 5}

In [87]:
########## groupby + 집계
result = Choice.objects.values("question").annotate(
    min=Min("votes"),
    max=Max("votes")
)
result

<QuerySet [{'question': 1, 'min': 0, 'max': 5}]>

### Join
- 자식 table(model) 기준으로 부모 table(model) data 조회
- `자식모델객체.FK_Field`

In [92]:
# Choice(자식) --> Question(부모)
c1 = Choice.objects.get(pk=1)
c1.pk, c1.id, c1.choice_text, c1.votes, c1.question

(1, 1, '짜장미엔', 3, <Question: 1. 가장 좋아하는 음식은 ?>)

In [93]:
# c1이 참조하는 question의 정보
c1.question.pk, c1.question.question_text, c1.question.pub_date

(1,
 '가장 좋아하는 음식은 ?',
 datetime.datetime(2025, 7, 7, 6, 31, 23, 893928, tzinfo=datetime.timezone.utc))

In [94]:
result_list = Choice.objects.filter(pk__lte=2)
# 조회한 choice의 질문 - 보기
for result in result_list:
    print(f"질문 : {result.question.question_text}, 조회한 보기 : {result.choice_text}")

질문 : 가장 좋아하는 음식은 ?, 조회한 보기 : 짜장미엔
질문 : 가장 좋아하는 음식은 ?, 조회한 보기 : 탕수육


- 부모 table(model) 기준으로 자식 table(model)을 조회
	- `부모모델객체.자식모델클래스이름(소문자)_set`을 통해서 부모객체를 참조하는 자식 date를 조회 가능

In [ ]:
q1 = Question.objects.get(pk = 1)
q1.pk, q1.question_text, q1.pub_date

(1,
 '가장 좋아하는 음식은 ?',
 datetime.datetime(2025, 7, 7, 6, 31, 23, 893928, tzinfo=datetime.timezone.utc))

In [97]:
# RelatedManager
# -> 부모 객체(q1)와 관련잇는 자식의 date 안에서만 조회할 수 있는 model manager
from random import choice


q1.choice_set

choice_list = q1.choice_set.all()
choice_list

<QuerySet [<Choice: 1. 짜장미엔>, <Choice: 2. 탕수육>, <Choice: 3. 탕수육>, <Choice: 4. 짬뽕>, <Choice: 5. 고추잡채>]>

In [98]:
print("질문 : ", q1.question_text)
print("보기")
for c in choice_list:
    print(f"{c.pk}. {c.choice_text}, {c.votes}")

질문 :  가장 좋아하는 음식은 ?
보기
1. 짜장미엔, 3
2. 탕수육, 10
3. 탕수육, 2
4. 짬뽕, 13
5. 고추잡채, 2


In [99]:
# 전체 질문을 조회하고 그것에 대해서 위 형식으로 출력.
q_list = Question.objects.all()
for q in q_list:
    print(f"{q.pk}. {q.question_text}")
    c_list = q.choice_set.all()
    for idx, c in enumerate(c_list, start = 1):
        print(f"\t{idx}. {c.choice_text} - {c.votes}")

1. 가장 좋아하는 음식은 ?
	1. 짜장미엔 - 3
	2. 탕수육 - 10
	3. 탕수육 - 2
	4. 짬뽕 - 13
	5. 고추잡채 - 2
2. 가장 좋아하는 날씨는 ?
	1. 화창~ - 0
	2. 비 - 4


### insert / update

- `모델객체.save()`
- 모델 객체의 pk가 DB에 없으면 insert, 있으면 update

In [100]:
# insert
## question : pk-id : 자동 증가 정수(생략), pub_date : insert할 때 일시를 추가(생략)
new_q = Question(question_text = "배우고 싶은 언어는 무엇입니까 ?")

print(new_q.question_text)
print(new_q.pk, new_q.pub_date)

배우고 싶은 언어는 무엇입니까 ?
None None


In [101]:
# save - pk : None(DB에 없는 data) ==> insert
new_q.save()

In [102]:
# insert 후 자동저장되는 값(pk, pub_date)이 모델객체의 field에 저장된다.
print(new_q.pk)
print(new_q.pub_date)

3
2025-07-08 11:55:30.748017+00:00


In [104]:
# update
q = Question.objects.get(pk = 3)
q.question_text = "여행으로 가고 싶은 나라를 선택해주세용 !"